# Gestión y Análisis de Datos — Fase 2

## Integración de Múltiples Fuentes y Enriquecimiento de Información

En la **Fase 1** trabajaste con un recurso a la vez: lo extrajiste, lo exploraste y lo limpiaste.
Esta fase da dos pasos más.

**Parte A — Integración de fuentes estructuradas.** El dato que necesitas ya no está en una sola
tabla, sino repartido entre varias unidas por claves. Vas a combinarlas con `merge`, viendo qué
pasa cuando eliges mal el tipo de unión.

**Parte B — Enriquecimiento con datos no estructurados.** Muchos huecos de la tabla no son
irrecuperables: la información existe, escrita en lenguaje natural, dentro de las transcripciones
de las llamadas de atención al cliente. Vas a sacarla con **expresiones regulares** y **spaCy**.

---

### Cómo se trabaja este notebook

El notebook alterna cuatro tipos de celda:

| | Qué es | Qué haces |
|---|---|---|
| ▶️ | **Ejecutar y observar** | Corres la celda y lees la evidencia que imprime |
| ✏️ | **`# COMPLETA`** | Escribes una o dos líneas para que la celda funcione |
| 🔍 | **Retos** | Escribes tú el bloque, con un esqueleto y pistas. Cortos, pero tienen truco |
| 📝 | **Anotaciones** | Respondes en 2-3 líneas qué concluyes de lo que acabas de ver |

Ninguna parte es difícil por separado, pero todas piden entender qué está pasando. **No se evalúa
la cantidad de código, sino el criterio con que interpretas lo que ves.**

---

**La pregunta que guía toda la fase:**
> *¿Por qué hay clientes que puntúan mal, y qué parte de eso depende de nosotros?*
> Ninguna fuente lo responde sola. La respuesta aparece al integrarlas.

**Evaluación:** Práctica 2 — 20% de la nota del corte.
**Fuente de datos:** `http://104.225.223.220:8003` — [documentación interactiva](http://104.225.223.220:8003/docs)

> ### 📓 Versión del estudiante
> Ejecuta las celdas en orden y **responde todas las casillas 📝** con lo que veas en las salidas.
> Las pocas líneas marcadas `# COMPLETA` las escribes tú. Al final (sección 20) están las tres
> tareas que debes añadir para completar la entrega.

## 0. Preparación del entorno

Además de lo de la Fase 1 necesitas **spaCy** y su modelo de español, que es el que reconoce
nombres de persona dentro de un texto. La descarga solo hace falta la primera vez.

In [2]:
!pip install -q pandas numpy matplotlib seaborn requests spacy
!python -m spacy download es_core_news_sm


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
     ----------------------------------- --- 11.8/12.9 MB 72.0 MB/s eta 0:00:01
     ---------------------------------------- 12.9/12.9 MB 59.0 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 170)
sns.set_theme(style='whitegrid')

BASE_URL = 'http://104.225.223.220:8003'

print('Entorno listo. Trabajando contra', BASE_URL)

Entorno listo. Trabajando contra http://104.225.223.220:8003


## 1. EXTRACT — Traer varias fuentes de una vez

Todos los endpoints responden con la misma envoltura:

```json
{ "total": 6000, "limit": 100, "offset": 0, "data": [ ... ] }
```

Los registros van en `data` y `total` dice cuántos hay **en el servidor**.

Eso último importa: la API no te deja pedir todo de golpe (cada endpoint tiene su tope) y
`detalle_pedido` tiene más filas que el tope máximo. Si pides y ya está, te llevas un trozo y no
te enteras. Por eso la función **pagina**: pide por bloques usando `offset` hasta juntar los
`total` registros. Es lo que hace cualquier proceso ETL real.

In [4]:
def obtener(recurso, tam_pagina=1000, **filtros):
    """Descarga un recurso COMPLETO de la API, paginando hasta traerlo todo."""
    filas, offset = [], 0
    while True:
        respuesta = requests.get(
            f'{BASE_URL}/api/{recurso}',
            params={'limit': tam_pagina, 'offset': offset, **filtros}, timeout=120)
        respuesta.raise_for_status()
        cuerpo = respuesta.json()
        # COMPLETA: añade a `filas` los registros de esta página (están en cuerpo['data'])
        filas.extend(cuerpo['data'])
        offset += tam_pagina
        if offset >= cuerpo['total']:
            return pd.DataFrame(filas)


# Comprobación: detalle_pedido tiene más filas que el tope de una sola petición
prueba = obtener('detalle_pedido')
print(f'detalle_pedido descargado completo: {len(prueba):,} filas')
prueba.head(3)

detalle_pedido descargado completo: 10,498 filas


,detalle_id,pedido_id,producto_id,cantidad,precio_unitario,descuento_porcentaje,subtotal
0,1,1,87,3,2878.52,0.0,8635.56
1,2,1,46,5,15.80,0.0,79.00
2,3,2,103,5,33.73,0.0,168.65


### 1.1 Las fuentes de esta fase

| Recurso | Qué contiene | Clave propia | Se une por |
|---|---|---|---|
| `usuarios` | quién compra | `usuario_id` | — |
| `direcciones` | dónde vive (varias por usuario) | `direccion_id` | `usuario_id` |
| `pedidos` | qué se pidió y cuándo | `pedido_id` | `usuario_id` |
| `detalle_pedido` | qué artículos lleva cada pedido | `detalle_id` | `pedido_id`, `producto_id` |
| `productos` | precio y categoría | `producto_id` | `categoria_id` |
| `envios` | transportista y fechas de entrega | `envio_id` | `pedido_id` |
| `resenas` | calificación y comentario | `resena_id` | `usuario_id`, `producto_id`, `pedido_id` |
| `llamadas` | **transcripción** de atención al cliente | `llamada_id` | `usuario_id` |

In [5]:
recursos = ['usuarios', 'direcciones', 'pedidos', 'detalle_pedido',
            'productos', 'categorias', 'envios', 'resenas', 'llamadas']

datos = {nombre: obtener(nombre) for nombre in recursos}

for nombre, df in datos.items():
    print(f'{nombre:16s} {df.shape[0]:>6,} filas × {df.shape[1]:>2} columnas')

usuarios          3,000 filas × 11 columnas
direcciones       4,525 filas ×  8 columnas
pedidos           6,000 filas × 12 columnas
detalle_pedido   10,498 filas ×  7 columnas
productos           120 filas ×  8 columnas
categorias           15 filas ×  4 columnas
envios            3,833 filas × 10 columnas
resenas           1,551 filas ×  8 columnas
llamadas          3,000 filas ×  7 columnas


In [6]:
usuarios    = datos['usuarios'].copy()
direcciones = datos['direcciones'].copy()
pedidos     = datos['pedidos'].copy()
detalle     = datos['detalle_pedido'].copy()
productos   = datos['productos'].copy()
categorias  = datos['categorias'].copy()
envios      = datos['envios'].copy()
resenas     = datos['resenas'].copy()
llamadas    = datos['llamadas'].copy()

print('Nueve fuentes cargadas en memoria.')

Nueve fuentes cargadas en memoria.


### 📝 Anotación 1

**Mira los tamaños de arriba. Hay 3.000 usuarios pero 4.525 direcciones y 6.000 pedidos. ¿Qué te dice eso sobre la relación entre esas tablas?**

_Responde en 2-3 líneas, mirando la salida de la celda anterior:_

> 

hay 3k usuarios pero cada uno de esos usuarios puede tener mas de 1 direccion asociada a su cuenta por eso hay 4525, ademas, un mismo usuario tambien puede realizar diferentes pedidos al mismo tiempo.

## 2. Mirar el modelo antes de unir nada

Un `merge` mal planteado **no da error**: da un resultado silenciosamente equivocado. Antes de la
primera unión conviene responder tres preguntas sobre cada par de tablas:

1. **¿Qué columna comparten?** → la clave
2. **¿Es única esa clave en cada lado?** → la cardinalidad (1:1 o 1:N)
3. **¿Todos los valores de un lado existen en el otro?** → cuántas filas vas a perder

La celda siguiente contesta las dos primeras de golpe.

In [7]:
def perfil_clave(df, columna, nombre):
    # COMPLETA: `unicos` = cuántos valores DISTINTOS tiene esa columna (método .nunique)
    total, unicos = len(df), df[columna].nunique()
    tipo = 'ÚNICA (1:1)' if unicos == total else 'SE REPITE (1:N)'
    print(f'{nombre:16s} {columna:14s} {total:>6,} filas | {unicos:>6,} únicos | {tipo}')


print('¿Puede cada columna actuar como clave?\n')
perfil_clave(usuarios,    'usuario_id', 'usuarios')
perfil_clave(direcciones, 'usuario_id', 'direcciones')
perfil_clave(pedidos,     'pedido_id',  'pedidos')
perfil_clave(pedidos,     'usuario_id', 'pedidos')
perfil_clave(detalle,     'pedido_id',  'detalle_pedido')
perfil_clave(envios,      'pedido_id',  'envios')
perfil_clave(resenas,     'pedido_id',  'resenas')

¿Puede cada columna actuar como clave?

usuarios         usuario_id      3,000 filas |  3,000 únicos | ÚNICA (1:1)
direcciones      usuario_id      4,525 filas |  3,000 únicos | SE REPITE (1:N)
pedidos          pedido_id       6,000 filas |  6,000 únicos | ÚNICA (1:1)
pedidos          usuario_id      6,000 filas |  2,599 únicos | SE REPITE (1:N)
detalle_pedido   pedido_id      10,498 filas |  6,000 únicos | SE REPITE (1:N)
envios           pedido_id       3,833 filas |  3,833 únicos | ÚNICA (1:1)
resenas          pedido_id       1,551 filas |  1,551 únicos | ÚNICA (1:1)


### 📝 Anotación 2

**Según esa tabla, ¿cuáles de estas uniones son seguras (1:1) y cuáles van a multiplicar filas?**

_Responde en 2-3 líneas, mirando la salida de la celda anterior:_

> 

Las uniones que son seguras son usuarios, pedidos, envios y resenas porque no se van a repetir en mas de una fila, pero las demas si pueden hacerlo porque un usuario puede tener mas de 1 direccion o realizar mas de 1 pedido a la misma direccion

## 3. El primer merge: `inner` frente a `left`

Unimos `pedidos` con `envios`. La clave es 1:1, así que no hay riesgo de multiplicar filas...
pero **no todos los pedidos tienen envío**: los que aún no salieron de bodega no están en esa
tabla. El tipo de unión decide qué haces con ellos.

In [8]:
pe_inner = pedidos.merge(envios, on='pedido_id', how='inner')
pe_left  = pedidos.merge(envios, on='pedido_id', how='left')

print(f'pedidos originales     : {len(pedidos):,}')
print(f'merge INNER con envios : {len(pe_inner):,}   ← se pierden {len(pedidos) - len(pe_inner):,} pedidos')
print(f'merge LEFT  con envios : {len(pe_left):,}   ← se conservan todos')
print()
print(f'Pedidos sin envío (NaN tras el left): {pe_left["transportista"].isna().sum():,}')

pedidos originales     : 6,000
merge INNER con envios : 3,833   ← se pierden 2,167 pedidos
merge LEFT  con envios : 6,000   ← se conservan todos

Pedidos sin envío (NaN tras el left): 2,167


In [9]:
# ¿Cambia la conclusión de negocio según el join elegido?
print(f'Ticket medio con INNER : {pe_inner["total"].mean():>10,.2f}')
print(f'Ticket medio con LEFT  : {pe_left["total"].mean():>10,.2f}')
print()
print('Estado de los pedidos que el INNER descarta:')
print(pe_left[pe_left['transportista'].isna()]['estado'].value_counts())

Ticket medio con INNER :   1,852.23
Ticket medio con LEFT  :   1,844.09

Estado de los pedidos que el INNER descarta:
estado
Cancelado     869
Entregado     666
En camino     243
Procesando    208
Devuelto      181
Name: count, dtype: int64


### 📝 Anotación 3

**El `inner` descarta más de 2.000 pedidos reales. ¿En qué caso sería la opción correcta y en cuál sería un error?**

_Responde en 2-3 líneas, mirando la salida de la celda anterior:_

> 

Seria la opcion correcta dependiendo de lo que se quiera analizar, por ejempo, para analizar que pedidos si tienen envio y seria incorrecto usarlo cuando se quiera analizar todos los pedidos realizados, es por esto que inner descarta mas de 2000 pedidos realizados

### 🔍 Reto 1 — Los otros dos joins

Ya viste `inner` y `left`. Faltan `right` y `outer`. Calcúlalos y responde en el `print`: ¿cuál da el mismo número de filas que otro que ya conoces, y por qué?

In [ ]:
# Pista: es la misma línea de antes cambiando how=
pe_right = pedidos.merge(envios, on='pedido_id', how='right')
pe_outer  = pedidos.merge(envios, on='pedido_id', how='outer')

print(f'INNER : {len(pe_inner):,}')
print(f'LEFT  : {len(pe_left):,}')
print(f'RIGHT : {len(pe_right):,}')
print(f'OUTER : {len(pe_outer):,}')
# ¿Cuáles dos coinciden? Escribe aquí tu explicación como comentario:
#las 2 que coinciden son inner y right porque todos los envios tienen un pedido asignado, tambien se podria decir que left
#y outher coinciden porque no hay envios sin pedidos

INNER : 3,833
LEFT  : 6,000
RIGHT : 3,833
OUTER : 6,000


## 4. La explosión 1:N — el error que no avisa

`detalle_pedido` tiene **una fila por artículo**, no por pedido. Si la unes tal cual con `pedidos`,
cada pedido se repite tantas veces como artículos tenga, y cualquier suma sobre `total` cuenta ese
pedido varias veces.

Ejecuta y mira la última línea.

In [11]:
# COMPLETA: une `pedidos` con `detalle` por 'pedido_id' usando how='inner'
merge_naif = pedidos.merge(detalle, on='pedido_id', how='inner')

print(f'pedidos           : {len(pedidos):,} filas')
print(f'detalle_pedido    : {len(detalle):,} filas')
print(f'pedidos ⨝ detalle : {len(merge_naif):,} filas ')
print()
print(f'Facturación REAL               : {pedidos["total"].sum():>16,.2f}')
print(f'Facturación tras el merge naíf : {merge_naif["total"].sum():>16,.2f}')
print(f'\nEstás inflando la cifra {merge_naif["total"].sum() / pedidos["total"].sum():.2f} veces.')

pedidos           : 6,000 filas
detalle_pedido    : 10,498 filas
pedidos ⨝ detalle : 10,498 filas 

Facturación REAL               :    11,064,554.65
Facturación tras el merge naíf :    24,279,800.27

Estás inflando la cifra 2.19 veces.


### 4.1 La red de seguridad: `validate`

`pandas` puede comprobar la cardinalidad por ti y **lanzar un error** si no es la que esperabas.
Convierte un fallo silencioso en uno ruidoso, que es justo lo que quieres.

In [12]:
ok = pedidos.merge(envios, on='pedido_id', how='left', validate='one_to_one')
print(f'validate="one_to_one" con envios → correcto, {len(ok):,} filas\n')

try:
    pedidos.merge(detalle, on='pedido_id', how='left', validate='one_to_one')
except Exception as error:
    print(f'validate="one_to_one" con detalle → {type(error).__name__}')
    print(f'   {error}')

validate="one_to_one" con envios → correcto, 6,000 filas

validate="one_to_one" con detalle → MergeError
   Merge keys are not unique in right dataset; not a one-to-one merge
Duplicates in right:
  pedido_id
         1
         2
         3
         5
         5 ...


### 📝 Anotación 4

**Con tus palabras: ¿qué acaba de detectar `validate` y por qué es peor un merge que "funciona" que uno que falla?**

_Responde en 2-3 líneas, mirando la salida de la celda anterior:_

> 

El validate del punto anterior detecto que las merge keys no son unicas despues de hacer un rigth, por lo tanto no se puede hacer un merge uno a uno de cada tabla. Es peor el merge que funciona incorrectamente porque el primero simplemente da un resultado relacionado a datos incorrectos mientras que el que falla permite analizar nuevamente para detectar el problema y hacerlo bien

## 5. Agregar antes de unir

La solución al 1:N es **subir la tabla de detalle al nivel del pedido** antes de unirla: un
`groupby` que convierta las N líneas en una sola fila de resúmenes. Después, la unión ya es 1:1.

In [14]:
detalle_por_pedido = detalle.groupby('pedido_id').agg(
    n_articulos         = ('detalle_id', 'count'),
    unidades_totales    = ('cantidad', 'sum'),
    # COMPLETA: suma de la columna 'subtotal'
    importe_articulos   = ('subtotal', 'sum'),
    # COMPLETA: cuántos 'producto_id' DISTINTOS lleva el pedido (nunique)
    productos_distintos = ('producto_id', 'nunique'),
).reset_index()

print(f'detalle_pedido : {len(detalle):,} filas (una por artículo)')
print(f'agregado       : {len(detalle_por_pedido):,} filas (una por pedido)')
detalle_por_pedido.head()

detalle_pedido : 10,498 filas (una por artículo)
agregado       : 6,000 filas (una por pedido)


,pedido_id,n_articulos,unidades_totales,importe_articulos,productos_distintos
0,1,2,8,8714.56,2
1,2,2,9,719.41,2
2,3,2,6,1426.68,2
3,4,1,5,240.20,1
4,5,4,14,1854.43,4


In [15]:
pedidos_full = (pedidos
                .merge(detalle_por_pedido, on='pedido_id', how='left', validate='one_to_one')
                .merge(envios,             on='pedido_id', how='left', validate='one_to_one'))

print(f'pedidos_full : {len(pedidos_full):,} filas (los mismos {len(pedidos):,} pedidos)')
print(f'Facturación  : {pedidos_full["total"].sum():,.2f}   ← ahora sí coincide con la real')
pedidos_full.head(3)

pedidos_full : 6,000 filas (los mismos 6,000 pedidos)
Facturación  : 11,064,554.65   ← ahora sí coincide con la real


,pedido_id,usuario_id,fecha_pedido,estado,subtotal,descuento,costo_envio,total,direccion_envio_id,sucursal_id,cupon_id,notas,n_articulos,unidades_totales,importe_articulos,productos_distintos,envio_id,sucursal_origen_id,tipo_envio,transportista,numero_seguimiento,fecha_envio,fecha_entrega_estimada,fecha_entrega_real,estado_envio
0,1,2062,2025-10-01 20:06:19,En camino,8714.56,0.0,8.37,8722.93,3108,5,NaN,Nihil ea neque delectus quos esse laudantium t...,2,8,8714.56,2,1.0,5.0,Express,Correos,TRK0000000001,2025-10-03,2025-10-05,NaN,En tránsito
1,2,478,2025-07-10 20:06:19,Entregado,719.41,0.0,9.15,728.56,695,12,NaN,Et rerum aliquam saepe quis dicta accusamus.,2,9,719.41,2,2.0,12.0,Express,DHL,TRK0000000002,2025-07-11,2025-07-12,2025-07-26,Entregado
2,3,1463,2026-03-21 20:06:19,Devuelto,1426.68,0.0,10.40,1437.08,2203,2,NaN,Molestias cupiditate velit incidunt.,2,6,1426.68,2,3.0,2.0,Estándar,Rappi,TRK0000000003,2026-03-22,2026-03-24,2026-03-30,En tránsito


### 5.1 Direcciones: mismo problema, otra solución

`direcciones` también es 1:N. Aquí no queremos agregar sino **quedarnos con una fila por usuario**:
la marcada como principal. Filtrar es la otra forma legítima de resolver un 1:N.

In [17]:
# COMPLETA: quédate solo con las filas cuya columna es_principal valga 1
principal = direcciones[direcciones['es_principal'] == 1].copy()

print(f'direcciones totales     : {len(direcciones):,}')
print(f'direcciones principales : {len(principal):,}')
print(f'¿usuario_id ya es único? {not principal["usuario_id"].duplicated().any()}')

principal = (principal[['usuario_id', 'ciudad', 'pais', 'codigo_postal']]
             .rename(columns={'ciudad': 'ciudad_envio', 'pais': 'pais_envio'}))
principal.head(3)

direcciones totales     : 4,525
direcciones principales : 3,000
¿usuario_id ya es único? True


,usuario_id,ciudad_envio,pais_envio,codigo_postal
0,1,Arequipa,Perú,36566
1,2,Cusco,Perú,06668-8433
2,3,Guadalajara,México,49014


### 📝 Anotación 5

**Has resuelto dos relaciones 1:N de formas distintas: `detalle` agregando y `direcciones` filtrando. ¿Qué determina cuál de las dos toca en cada caso?**

_Responde en 2-3 líneas, mirando la salida de la celda anterior:_

> 

Termina dependiendo de la informacion que se necesite mostrar, por ejemplo, con groupby se agrupa la informacion pero sigue estando visible, mientras que con el filtrado se "oculta" la demas informacion por lo que se estarian mostrando menos campos

## 6. Auditoría: ¿encajan de verdad las fuentes?

Antes de dar por buena la integración conviene comprobar que las claves foráneas apuntan a algo
que existe (**huérfanos**) y cuánta cobertura tiene cada fuente sobre las demás.

In [18]:
def auditar(hijo, col_hijo, padre, col_padre, etiqueta):
    claves_padre = set(padre[col_padre])
    # COMPLETA: cuántas filas del hijo tienen una clave que NO está en claves_padre
    #           (pista: .isin(...) dice cuáles SÍ están, y ~ invierte la condición)
    huerfanos = (~hijo[col_hijo].isin(claves_padre)).sum()
    cobertura = hijo[col_hijo].nunique() / len(claves_padre) * 100
    print(f'{etiqueta:34s} huérfanos: {huerfanos:>5,} | cubre el {cobertura:5.1f}% del padre')


auditar(pedidos,  'usuario_id', usuarios,  'usuario_id', 'pedidos → usuarios')
auditar(detalle,  'pedido_id',  pedidos,   'pedido_id',  'detalle → pedidos')
auditar(detalle,  'producto_id', productos, 'producto_id', 'detalle → productos')
auditar(envios,   'pedido_id',  pedidos,   'pedido_id',  'envios → pedidos')
auditar(resenas,  'pedido_id',  pedidos,   'pedido_id',  'resenas → pedidos')
auditar(llamadas, 'usuario_id', usuarios,  'usuario_id', 'llamadas → usuarios')

pedidos → usuarios                 huérfanos:     0 | cubre el  86.6% del padre
detalle → pedidos                  huérfanos:     0 | cubre el 100.0% del padre
detalle → productos                huérfanos:     0 | cubre el 100.0% del padre
envios → pedidos                   huérfanos:     0 | cubre el  63.9% del padre
resenas → pedidos                  huérfanos:     0 | cubre el  25.9% del padre
llamadas → usuarios                huérfanos:     0 | cubre el 100.0% del padre


### 📝 Anotación 6

**No hay huérfanos, pero las coberturas son muy distintas. ¿Qué significa que `resenas` cubra un porcentaje bajo de los pedidos?**

_Responde en 2-3 líneas, mirando la salida de la celda anterior:_

> 

El hecho de que el proncentaje de resenas sea tan bajo para los pedidos probablemente se puede deber a que los usuarios no dejan resenas de los pedidos que hacen, si se orserva la celda anterior a la anotacion 2 vemos que las resenas tienen 1551 filas para los 6k pedidos presentes y si se hace el calculo las cifras y porcentajes coinciden casi exactamente 6000 * 25/9% = ~1554

## 7. Nulos: cuatro cosas distintas con la misma cara

Tras integrar aparecen muchos `NaN`, y tratarlos todos igual es un error de análisis. Hay al
menos cuatro tipos y **solo uno es un problema de calidad**:

| Tipo | Qué pasó | Qué hacer |
|---|---|---|
| **No capturado** | el dato debía existir y no está | imputar, o recuperarlo de otra fuente |
| **No aplica** | no existe por diseño | dejarlo nulo, nunca imputar |
| **Aún no ocurre** | el evento no ha pasado todavía | excluir del cálculo |
| **NaN de join** | lo generó tu propio `left join` | depende de la pregunta |

In [ ]:
print('usuarios — nulos por columna:')
nulos_u = usuarios.isna().sum()
print((nulos_u[nulos_u > 0].to_frame('nulos')
       .assign(pct=lambda d: (d['nulos'] / len(usuarios) * 100).round(1))))

print(f'\npedidos — cupon_id nulo : {pedidos["cupon_id"].isna().sum():,} '
      f'({pedidos["cupon_id"].isna().mean() * 100:.1f}%)')

no_entregados = envios[envios['estado_envio'] != 'Entregado']
print(f'envios  — no entregados : {len(no_entregados):,} '
      f'(de los cuales {no_entregados["fecha_entrega_real"].isna().sum():,} sin fecha real)')

### 📝 Anotación 7

**Clasifica estos tres casos en la tabla de arriba: (a) el 20% de usuarios sin `nombre`, (b) el 80% de pedidos sin `cupon_id`, (c) los envíos sin `fecha_entrega_real`.**

_Responde en 2-3 líneas, mirando la salida de la celda anterior:_

> 

## 8. Inconsistencias entre columnas

Un dato puede estar presente, tener el tipo correcto y aun así ser **imposible**. Estas
contradicciones solo se ven cruzando dos columnas o dos fuentes.

Hay tres anomalías escondidas en `envios`. La celda las busca.

In [ ]:
env = envios.copy()
for col in ['fecha_envio', 'fecha_entrega_estimada', 'fecha_entrega_real']:
    env[col] = pd.to_datetime(env[col], errors='coerce')

contradictorios = env[(env['estado_envio'] != 'Entregado') & env['fecha_entrega_real'].notna()]
# COMPLETA: envíos cuya fecha_entrega_real sea ANTERIOR a la fecha_envio
imposibles      = env[...]
entregados_sin  = env[(env['estado_envio'] == 'Entregado') & env['fecha_entrega_real'].isna()]

print(f'A) Estado NO "Entregado" pero CON fecha de entrega real : {len(contradictorios):>4,}')
print(f'B) Entrega real ANTERIOR a la fecha de envío            : {len(imposibles):>4,}')
print(f'C) Estado "Entregado" pero SIN fecha de entrega real    : {len(entregados_sin):>4,}')

contradictorios[['envio_id', 'estado_envio', 'fecha_envio',
                 'fecha_entrega_estimada', 'fecha_entrega_real']].head()

### 📝 Anotación 8

**Elige UNA de las tres anomalías y decide qué harías con esas filas: ¿corregirlas, eliminarlas o marcarlas y seguir? Justifica en una línea.**

_Responde en 2-3 líneas, mirando la salida de la celda anterior:_

> 

## 9. Variables que solo existen al integrar

Lo valioso de unir fuentes no es tener más columnas, sino poder **construir las que ninguna
fuente contenía**. `dias_retraso` es el ejemplo perfecto: no está en ninguna tabla; aparece al
restar dos fechas, y solo tiene sentido en los envíos ya entregados.

In [ ]:
pf = pedidos_full.copy()
for col in ['fecha_pedido', 'fecha_envio', 'fecha_entrega_estimada', 'fecha_entrega_real']:
    pf[col] = pd.to_datetime(pf[col], errors='coerce')

entregado = pf['estado_envio'] == 'Entregado'

pf['dias_retraso'] = np.where(
    entregado & pf['fecha_entrega_real'].notna(),
    # COMPLETA: días de diferencia entre la entrega real y la estimada
    #           (resta las dos columnas y usa .dt.days)
    ...,
    np.nan)

pf['llego_tarde']  = np.where(pf['dias_retraso'].notna(), pf['dias_retraso'] > 0, np.nan)
pf['ticket_medio_articulo'] = pf['total'] / pf['n_articulos']

print(f'Pedidos entregados con retraso medible : {pf["dias_retraso"].notna().sum():,}')
print(f'De ellos, llegaron TARDE               : {(pf["dias_retraso"] > 0).sum():,} '
      f'({(pf["dias_retraso"] > 0).mean() * 100:.1f}%)')
print(f'Retraso medio cuando hay retraso       : {pf.loc[pf["dias_retraso"] > 0, "dias_retraso"].mean():.1f} días')

### 9.1 ¿Todos los transportistas son iguales?

Primera conclusión de negocio del taller. Ejecuta y mira el orden de la tabla.

In [ ]:
por_transportista = (pf[pf['dias_retraso'].notna()]
                     .groupby('transportista')
                     .agg(envios=('envio_id', 'count'),
                          pct_tarde=('llego_tarde', 'mean'),
                          retraso_medio=('dias_retraso', 'mean'))
                     .sort_values('pct_tarde', ascending=False))
por_transportista['pct_tarde'] = (por_transportista['pct_tarde'] * 100).round(1)
por_transportista['retraso_medio'] = por_transportista['retraso_medio'].round(2)
print(por_transportista)

por_transportista['pct_tarde'].plot(kind='barh', figsize=(8, 3.5), color='#6B4FBB')
plt.xlabel('% de envíos entregados con retraso')
plt.ylabel('')
plt.title('Puntualidad por transportista')
plt.tight_layout()
plt.show()

### 📝 Anotación 9

**Con esta tabla delante: si pudieras dejar de trabajar con un transportista, ¿con cuál? ¿Qué dato adicional pedirías antes de decidirlo de verdad?**

_Responde en 2-3 líneas, mirando la salida de la celda anterior:_

> 

### 🔍 Reto 2 — La misma receta, otra pregunta

Acabas de comparar transportistas. Repite exactamente el mismo análisis pero agrupando por `tipo_envio` (Express, Estándar…). ¿Pagar por un envío Express te libra del retraso?

In [ ]:
# Pista: copia la celda anterior y cambia SOLO la columna del groupby
por_tipo = ...
print(por_tipo)

## 10. ¿El retraso explica la mala calificación?

Aquí se juntan tres fuentes: `resenas` → `pedidos` → `envios`. Ninguna de las tres, por separado,
puede responder la pregunta.

In [ ]:
resenas_env = resenas.merge(
    pf[['pedido_id', 'transportista', 'dias_retraso', 'llego_tarde', 'estado_envio']],
    on='pedido_id', how='left', validate='many_to_one')

comparables = resenas_env[resenas_env['dias_retraso'].notna()]

resumen = (comparables.groupby('llego_tarde')['calificacion']
           .agg(['count', 'mean']).round(2)
           .rename(index={0.0: 'Envío puntual', 1.0: 'Envío con retraso'}))
print(resumen)

diferencia = (resumen.loc['Envío puntual', 'mean'] - resumen.loc['Envío con retraso', 'mean'])
print(f'\nUn envío con retraso cuesta {diferencia:.2f} estrellas de media.')

In [ ]:
sns.boxplot(data=comparables, x='llego_tarde', y='calificacion',
            hue='llego_tarde', palette=['#6B4FBB', '#C0392B'], legend=False)
plt.xticks([0, 1], ['Puntual', 'Con retraso'])
plt.xlabel('')
plt.ylabel('Calificación de la reseña')
plt.title('La experiencia de envío se refleja en la reseña')
plt.tight_layout()
plt.show()

### 📝 Anotación 10

**Acabas de encontrar una relación entre retraso y calificación. Nombra al menos una razón por la que esto NO demuestra todavía que el retraso *cause* la mala nota.**

_Responde en 2-3 líneas, mirando la salida de la celda anterior:_

> 

## 11. Cambiar de granularidad: una fila por cliente

Hasta ahora la unidad de análisis era el pedido. Para perfilar clientes hay que **subir** a
usuario, y eso siempre es un `groupby` + un `merge`.

In [ ]:
por_usuario = pf.groupby('usuario_id').agg(
    n_pedidos     = ('pedido_id', 'count'),
    gasto_total   = ('total', 'sum'),
    # COMPLETA: gasto MEDIO por pedido de ese usuario
    ticket_medio  = ...,
    ultima_compra = ('fecha_pedido', 'max'),
    pedidos_tarde = ('llego_tarde', 'sum'),
).reset_index()

rating = resenas.groupby('usuario_id')['calificacion'].mean().reset_index(name='rating_medio')

clientes = (usuarios
            .merge(principal,   on='usuario_id', how='left', validate='one_to_one')
            .merge(por_usuario, on='usuario_id', how='left', validate='one_to_one')
            .merge(rating,      on='usuario_id', how='left', validate='one_to_one'))

print(f'clientes: {len(clientes):,} filas (una por usuario) × {clientes.shape[1]} columnas')
print(f'Usuarios sin ningún pedido: {clientes["n_pedidos"].isna().sum():,}')
clientes.head(3)

### 11.1 Un hueco de código

Los usuarios sin pedidos tienen `NaN` en `n_pedidos` y `gasto_total`. Aquí ese `NaN` sí significa
cero: no han comprado nada. Completa la línea.

In [ ]:
# COMPLETA: rellena con 0 los nulos de esas tres columnas (usa .fillna)
clientes[['n_pedidos', 'gasto_total', 'pedidos_tarde']] = ...

print(clientes[['n_pedidos', 'gasto_total', 'pedidos_tarde']].isna().sum())
print(f'\nClientes sin compras: {(clientes["n_pedidos"] == 0).sum():,}')

### 📝 Anotación 11

**`rating_medio` también tiene NaN y NO lo hemos rellenado con 0. ¿Por qué sería un error hacerlo?**

_Responde en 2-3 líneas, mirando la salida de la celda anterior:_

> 

### 🔍 Reto 3 — Define tú qué es un cliente VIP

No hay una definición correcta: la eliges tú. Crea la columna booleana `es_vip` con **tu** criterio (gasto, número de pedidos, antigüedad, o una mezcla) y cuenta cuántos salen. Después mira si tu grupo VIP puntúa mejor o peor que el resto: ese contraste es lo interesante.

In [ ]:
# Pista: define primero un umbral (por ejemplo con .quantile(0.90)) y luego la condición
umbral_gasto = ...
clientes['es_vip'] = ...

print(f'Clientes VIP: {clientes["es_vip"].sum():,} de {len(clientes):,}')
print(clientes.groupby('es_vip')[['gasto_total', 'n_pedidos', 'rating_medio']].mean().round(2))

---
# Parte B — Enriquecimiento con datos no estructurados

Hasta aquí has trabajado con tablas. Pero en la sección 7 quedó un problema abierto: **621
usuarios sin nombre y 652 sin email**. En la Fase 1 la única salida era imputar o descartar.

Ahora tienes una fuente más: `llamadas`, con 3.000 transcripciones de atención al cliente. Y en
esas conversaciones los clientes **dicen su nombre en voz alta**.

La información no se perdió. Solo está escrita en lenguaje natural.

## 12. Mirar el texto antes de procesarlo

Regla de oro del procesamiento de texto: **lee unos cuantos ejemplos a mano antes de escribir la
primera expresión regular**. Los patrones se descubren mirando, no adivinando.

In [ ]:
print(llamadas[['llamada_id', 'usuario_id', 'tipo_llamada', 'duracion_segundos']].head())
print('\nMotivos de llamada:')
print(llamadas['tipo_llamada'].value_counts())

In [ ]:
ejemplo = llamadas[llamadas['tipo_llamada'] == 'consulta_pedido'].iloc[0]
print(f"Llamada {ejemplo['llamada_id']} — usuario {ejemplo['usuario_id']}\n")
print(ejemplo['transcripcion'])

### 📝 Anotación 12

**Lee la transcripción de arriba. Señala al menos tres datos concretos que aparecen en el texto y que podrían servir para rellenar o enriquecer las tablas.**

_Responde en 2-3 líneas, mirando la salida de la celda anterior:_

> 

## 13. Extracción con expresiones regulares

Una **regex** describe la *forma* de un dato. Es la herramienta correcta cuando lo que buscas
tiene una estructura predecible: un email siempre lleva `@`, un código de pedido siempre es
`PED-` seguido de seis dígitos.

| Patrón | Qué significa |
|---|---|
| `\d` | un dígito | 
| `\w` | letra, dígito o guion bajo |
| `+` | uno o más del elemento anterior |
| `{6}` | exactamente seis |
| `( )` | **grupo de captura**: la parte que quieres quedarte |
| `?:` | grupo que agrupa pero no captura |

In [ ]:
PATRONES = {
    'codigo_pedido': re.compile(r'PED-(\d{6})'),
    'email':         re.compile(r'([\w.+-]+@[\w-]+\.[\w.]+)'),
    'telefono':      re.compile(r'(\+?\d[\d ().-]{7,}\d)'),
    # COMPLETA: importes con el formato $1.234,56
    #   pistas → \$ escapa el símbolo del dólar · \s? un espacio opcional
    #            [\d.]+ dígitos y puntos · ,\d{2} la coma y dos decimales
    'importe':       re.compile(r'...'),
}


def extraer(texto, patron):
    """Devuelve el primer grupo capturado, o None si el patrón no aparece."""
    encontrado = PATRONES[patron].search(texto)
    return encontrado.group(1) if encontrado else None


# Probamos sobre una sola transcripción antes de lanzarlo sobre 3.000
for nombre in PATRONES:
    print(f'{nombre:15s} → {extraer(ejemplo["transcripcion"], nombre)}')

In [ ]:
extraido = llamadas[['llamada_id', 'usuario_id', 'tipo_llamada']].copy()

for nombre in PATRONES:
    extraido[nombre] = llamadas['transcripcion'].apply(lambda t: extraer(t, nombre))

print('Cobertura de cada patrón sobre las 3.000 transcripciones:\n')
for nombre in PATRONES:
    n = extraido[nombre].notna().sum()
    print(f'  {nombre:15s} {n:>5,} de {len(extraido):,}  ({n / len(extraido) * 100:5.1f}%)')

extraido.head()

### 📝 Anotación 13

**Las coberturas son muy distintas entre sí. ¿Por qué `email` aparece solo en una quinta parte de las llamadas, mientras que `codigo_pedido` aparece en la mayoría?**

_Responde en 2-3 líneas, mirando la salida de la celda anterior:_

> 

### 🔍 Reto 4 — Escribe tu propio patrón

En las llamadas de facturación el agente dicta la **referencia del pago**, con el formato `PAY` seguido de 8 dígitos. No está en el diccionario `PATRONES`. Escríbelo, mide su cobertura y —lo importante— comprueba contra la tabla `pagos` que las referencias que extraes existen de verdad.

In [ ]:
# Pista: PAY en literal + 8 dígitos. Recuerda el paréntesis del grupo de captura.
PATRONES['referencia_pago'] = re.compile(r'...')
extraido['referencia_pago'] = llamadas['transcripcion'].apply(
    lambda t: extraer(t, 'referencia_pago'))

hallados = extraido['referencia_pago'].dropna()
pagos = obtener('pagos')
reales = set(pagos['referencia'])

print(f'Referencias extraídas : {len(hallados):,}')
# COMPLETA: ¿qué porcentaje de las extraídas existe de verdad en `reales`? (usa .isin)
print(f'Existen en la tabla pagos: ...')

## 14. El puente oculto entre las dos partes del taller

Fíjate en algo importante: **`llamadas` no tiene ninguna columna `pedido_id`**. En el modelo
relacional, las llamadas y los pedidos no están conectados.

Pero el código de pedido *sí* está en el texto. Acabas de extraerlo. Con eso puedes construir una
unión que **no existía en los datos**.

In [ ]:
print('Columnas de llamadas:', list(llamadas.columns))
print('\n¿Existe pedido_id?', 'pedido_id' in llamadas.columns)

In [ ]:
puente = extraido[extraido['codigo_pedido'].notna()].copy()
puente['pedido_id'] = puente['codigo_pedido'].astype(int)

llamadas_pedidos = puente.merge(
    pedidos[['pedido_id', 'usuario_id', 'total', 'estado']].rename(
        columns={'usuario_id': 'usuario_del_pedido'}),
    on='pedido_id', how='left')

print(f'Llamadas con código extraído : {len(llamadas_pedidos):,}')
print(f'Códigos que NO existen como pedido: {llamadas_pedidos["total"].isna().sum():,}')

coincide = (llamadas_pedidos['usuario_id'] == llamadas_pedidos['usuario_del_pedido'])
print(f'\nVerificación: el pedido mencionado pertenece a quien llama en '
      f'{coincide.sum():,} de {len(llamadas_pedidos):,} casos ({coincide.mean() * 100:.1f}%)')

### 📝 Anotación 14

**Acabas de comprobar que el pedido extraído del texto pertenece de verdad al usuario que llamaba. ¿Por qué esa comprobación es imprescindible antes de usar un dato extraído con regex?**

_Responde en 2-3 líneas, mirando la salida de la celda anterior:_

> 

## 15. Recuperar los nombres que faltan

Los clientes se presentan de varias formas: *"me llamo X"*, *"soy X"*, *"Con X, buen día"*. Todas
comparten un patrón: van en una línea que empieza por `Cliente:` y les sigue un nombre propio
(dos palabras con mayúscula inicial).

In [ ]:
PATRON_NOMBRE = re.compile(
    r'Cliente:.*?(?:me llamo|soy|Con)\s+'
    r'([A-ZÁÉÍÓÚÑ][a-záéíóúñ]+(?:\s+[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+)+)')

# COMPLETA: aplica PATRON_NOMBRE a cada transcripción y quédate con el grupo capturado.
#           Ojo: .search() devuelve None cuando no encuentra nada, hay que contemplarlo.
extraido['nombre_regex'] = llamadas['transcripcion'].apply(
    lambda t: ...)

encontrados = extraido['nombre_regex'].notna().sum()
print(f'Nombres extraídos por regex: {encontrados:,} de {len(extraido):,} '
      f'({encontrados / len(extraido) * 100:.1f}%)')
extraido[['usuario_id', 'nombre_regex']].head()

### 15.1 ¿Es correcto lo que extrajo?

No basta con que la regex encuentre *algo*. Hay que contrastarlo con los usuarios cuyo nombre
**sí** conocemos: es la única forma de medir la precisión.

In [ ]:
control = extraido[['usuario_id', 'nombre_regex']].merge(
    usuarios[['usuario_id', 'nombre', 'apellido']], on='usuario_id', how='left')

completos = control[control['nombre'].notna() & control['apellido'].notna()
                    & control['nombre_regex'].notna()].copy()
completos['nombre_tabla'] = completos['nombre'] + ' ' + completos['apellido']
aciertos = (completos['nombre_regex'] == completos['nombre_tabla']).mean()

print(f'Casos de control (nombre y apellido conocidos): {len(completos):,}')
print(f'La regex acierta el nombre completo en el {aciertos * 100:.1f}% de ellos')

print('\nEjemplos donde NO coincide:')
print(completos[completos['nombre_regex'] != completos['nombre_tabla']]
      [['nombre_regex', 'nombre_tabla']].head())

### 📝 Anotación 15

**Mira los ejemplos que "no coinciden". ¿Son de verdad errores de la regex?**

_Responde en 2-3 líneas, mirando la salida de la celda anterior:_

> 

## 16. Cuando la regex no llega: spaCy

Una regex reconoce formas fijas. Pero un nombre propio **no tiene forma fija**: se identifica por
el contexto de la frase. Para eso está el **Reconocimiento de Entidades Nombradas (NER)**, que
spaCy hace con un modelo entrenado en español.

Las etiquetas que nos interesan:

| Etiqueta | Qué reconoce |
|---|---|
| `PER` | personas |
| `LOC` | lugares |
| `ORG` | organizaciones y empresas |

spaCy es mucho más lento que una regex, así que lo aplicamos **solo donde la regex falló**. Esa
es la idea de la *cascada*.

In [ ]:
import spacy

nlp = spacy.load('es_core_news_sm')

doc = nlp(ejemplo['transcripcion'])
print('Entidades detectadas en la transcripción de ejemplo:\n')
for entidad in doc.ents:
    print(f'  {entidad.label_:5s}  {entidad.text}')

In [ ]:
faltan = extraido[extraido['nombre_regex'].isna()]
print(f'Transcripciones donde la regex NO encontró nombre: {len(faltan):,}')
print('spaCy se aplica solo a esas (por eso tarda segundos y no minutos).\n')


def primera_persona(texto):
    """Primer PER que spaCy encuentre en una línea del cliente."""
    lineas = [l for l in texto.split('\n') if l.startswith('Cliente:')]
    for linea in lineas:
        for entidad in nlp(linea).ents:
            # COMPLETA: solo nos interesan las entidades de tipo persona
            if entidad.label_ == ...:
                return entidad.text
    return None


recuperados = {idx: primera_persona(llamadas.loc[idx, 'transcripcion']) for idx in faltan.index}
extraido.loc[faltan.index, 'nombre_ner'] = pd.Series(recuperados)

print(f'spaCy recuperó {extraido["nombre_ner"].notna().sum():,} nombres adicionales.')
extraido.loc[faltan.index, ['usuario_id', 'nombre_regex', 'nombre_ner']].head(10)

### 📝 Anotación 16

**spaCy recupera bastantes menos casos que la regex y tarda mucho más. ¿Por qué entonces merece la pena tenerlo en el pipeline?**

_Responde en 2-3 líneas, mirando la salida de la celda anterior:_

> 

## 17. Estrategia de confianza y trazabilidad

Combinamos ambos métodos en cascada, de más fiable a menos, y **dejamos constancia del origen de
cada dato**. Esa columna `origen_dato` es lo que separa un enriquecimiento serio de inventarse
datos: cualquiera puede auditar después de dónde salió cada valor.

In [ ]:
extraido['nombre_final'] = extraido['nombre_regex'].fillna(extraido.get('nombre_ner'))

extraido['origen_dato'] = np.select(
    [extraido['nombre_regex'].notna(), extraido['nombre_final'].notna()],
    # COMPLETA: las dos etiquetas, en el MISMO orden que las condiciones de arriba
    [..., ...],
    default='no_recuperado')

print(extraido['origen_dato'].value_counts())
print(f'\nCobertura total: {extraido["nombre_final"].notna().mean() * 100:.1f}% de las llamadas')

## 18. Devolver lo extraído a la tabla

Último paso del ciclo: llevar lo aprendido del texto a la tabla de clientes. Dos reglas:

1. Se rellena **solo donde había un hueco** (`fillna`). Nunca se sobrescribe un dato que la fuente
   estructurada ya declaraba: el texto es una fuente secundaria.
2. Se conserva la columna de origen, para que quien lea el dataset sepa qué es dato original y qué
   es reconstrucción.

In [ ]:
por_usuario_texto = (extraido.sort_values('origen_dato')
                     .groupby('usuario_id')
                     .agg(nombre_texto=('nombre_final', 'first'),
                          email_texto=('email', 'first'),
                          telefono_texto=('telefono', 'first'),
                          origen_dato=('origen_dato', 'first'))
                     .reset_index())

clientes_enriquecidos = clientes.merge(por_usuario_texto, on='usuario_id',
                                       how='left', validate='one_to_one')

antes = pd.Series({
    'nombre':   (clientes_enriquecidos['nombre'].isna()
                 | clientes_enriquecidos['apellido'].isna()).sum(),   # ficha incompleta
    'email':    clientes_enriquecidos['email'].isna().sum(),
    'telefono': clientes_enriquecidos['telefono'].isna().sum(),
})

# La tabla puede traer el nombre COMPLETO, solo una de las dos partes, o nada.
# Solo damos por bueno lo estructurado cuando están las dos partes; si está a
# medias, el texto aporta la ficha completa (no sobrescribimos: completamos).
nombre_tabla = ((clientes_enriquecidos['nombre'].fillna('') + ' ' +
                 clientes_enriquecidos['apellido'].fillna('')).str.strip()
                .replace('', np.nan))
completo_en_tabla = clientes_enriquecidos['nombre'].notna() & clientes_enriquecidos['apellido'].notna()

clientes_enriquecidos['nombre_completo'] = (
    nombre_tabla.where(completo_en_tabla)          # ficha completa en la tabla
    .fillna(clientes_enriquecidos['nombre_texto']) # si no, la del texto
    .fillna(nombre_tabla))                         # y si el texto tampoco, lo que hubiera

print(f'Fichas completas ya en la tabla : {completo_en_tabla.sum():,}')
print(f'Fichas a medias completadas con el texto : '
      f'{(~completo_en_tabla & clientes_enriquecidos["nombre_texto"].notna()).sum():,}')


# COMPLETA: rellena SOLO los huecos de 'email' con lo extraído en 'email_texto' (.fillna)
clientes_enriquecidos['email']    = ...
clientes_enriquecidos['telefono'] = clientes_enriquecidos['telefono'].fillna(clientes_enriquecidos['telefono_texto'])

despues = pd.Series({
    'nombre':   clientes_enriquecidos['nombre_completo'].isna().sum(),
    'email':    clientes_enriquecidos['email'].isna().sum(),
    'telefono': clientes_enriquecidos['telefono'].isna().sum(),
})

impacto = pd.DataFrame({'incompletos_antes': antes.values, 'incompletos_despues': despues.values},
                       index=['nombre', 'email', 'telefono'])
impacto['recuperados'] = impacto['incompletos_antes'] - impacto['incompletos_despues']
impacto['%_recuperado'] = (impacto['recuperados'] / impacto['incompletos_antes'] * 100).round(1)
print(impacto)

### 📝 Anotación 17

**Mira la columna `%_recuperado`. En la Fase 1 la única opción ante estos nulos era imputar o descartar filas. ¿Qué cambia ahora, y por qué es mejor?**

_Responde en 2-3 líneas, mirando la salida de la celda anterior:_

> 

### 🔍 Reto 5 — La ficha del cliente

Cierre del taller: junta todo lo construido en una sola función. Dado un `usuario_id`, que imprima quién es (y **de dónde salió su nombre**), cuánto ha comprado, cómo puntúa y por qué llamó. Pruébala con un usuario cuyo nombre haya sido recuperado del texto: verás en una pantalla el resultado de las dos mitades del taller.

In [ ]:
# Pista: filtra clientes_enriquecidos por usuario_id, coge .iloc[0] y ve imprimiendo campos.
def ficha(usuario_id):
    fila = ...
    print(f"Cliente #{usuario_id}: {fila['nombre_completo']}  (origen: {fila['origen_dato']})")
    # COMPLETA: añade al menos tres líneas más (compras, rating, motivo de sus llamadas)


# Un usuario cuyo nombre NO estaba en la tabla y sí recuperamos del texto:
recuperado = clientes_enriquecidos[
    clientes_enriquecidos['nombre'].isna()
    & clientes_enriquecidos['nombre_completo'].notna()
    & (clientes_enriquecidos['n_pedidos'] > 0)].iloc[0]
ficha(int(recuperado['usuario_id']))

## 19. LOAD — Exportar el resultado

La última letra del ETL. El dataset que exportes aquí es parte de tu entrega.

In [ ]:
columnas = ['usuario_id', 'nombre_completo', 'email', 'telefono', 'edad', 'genero',
            'pais', 'ciudad_envio', 'estado_cuenta', 'fecha_registro',
            'n_pedidos', 'gasto_total', 'ticket_medio', 'pedidos_tarde',
            'rating_medio', 'origen_dato']
final = clientes_enriquecidos[[c for c in columnas if c in clientes_enriquecidos.columns]]

final.to_csv('fase2_clientes_enriquecido.csv', index=False, encoding='utf-8')
print(f'Exportado: fase2_clientes_enriquecido.csv → {final.shape[0]:,} filas × {final.shape[1]} columnas')
final.head()

---
## 20. Tu entrega — Práctica 2 (20%)

Este notebook, ejecutado de arriba abajo y **con todas las casillas 📝 rellenas**, es la mayor
parte de la entrega. Para completarla, añade debajo de esta celda las tres cosas siguientes.

### A. Una exploración propia (obligatoria)

Elige **una** de estas preguntas y respóndela con código, apoyándote en lo que ya construiste.
Bastan entre 5 y 15 líneas y un gráfico.

1. ¿Los clientes que han sufrido más retrasos gastan menos que el resto?
2. ¿Hay países o ciudades donde los envíos van sistemáticamente peor?
3. ¿Qué categorías de producto concentran las peores calificaciones?
4. ¿Los clientes que llaman por `queja` tienen un perfil de compra distinto?
5. ¿El importe del pedido influye en que llegue tarde o a tiempo?

### B. Un dato nuevo extraído del texto (obligatoria)

Vuelve a las transcripciones o a los comentarios de `resenas` y extrae **un dato que este notebook
no haya extraído todavía** (el transportista mencionado, el producto del que se queja el cliente,
el importe reclamado, el motivo de la devolución…). Mide su cobertura y comprueba su fiabilidad
contra la fuente estructurada, como hicimos en la sección 14.

### C. Tus conclusiones (media página, no más)

Cierra con:

- **Dos hallazgos** de negocio que hayas encontrado, con el número que los respalda.
- **Una decisión de preprocesamiento** que tomaste y por qué elegiste esa y no la alternativa.
- **Una limitación** de tu análisis: algo que tus datos *no* permiten afirmar.

---

### Qué se evalúa

| Criterio | Peso |
|---|---|
| Notebook ejecutado sin errores y anotaciones 📝 completas | 40% |
| Exploración propia (A) correcta y bien interpretada | 20% |
| Extracción nueva desde texto (B), con su verificación | 20% |
| Conclusiones (C): claras, apoyadas en datos y honestas sobre sus límites | 20% |

**Formato de entrega:** este `.ipynb` con las salidas visibles + el CSV exportado en la sección 19.